# **LangChain RAG (Retrieval-Augmented Generation) – In‑Depth Essential Codes**
This notebook demonstrates a **complete RAG pipeline using LangChain**.

Topics Covered:
- Document Loading
- Text Splitting
- Embeddings
- Vector Stores
- Retrievers
- Basic RAG Pipeline
- LCEL RAG Pipeline
- RAG with Prompt Templates
- RAG with Sources
- Conversational RAG
- Streaming RAG responses


## **1. Install Required Libraries**

In [ ]:
!pip install langchain langchain-openai langchain-community faiss-cpu chromadb tiktoken

## **2. Load Environment Variables**

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.getenv('OPENAI_API_KEY')

## **3. Load Documents**

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('sample.txt')

documents = loader.load()

documents[:1]

## **4. Split Documents into Chunks**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

len(chunks)

## **5. Create Embeddings**

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()


## **6. Create Vector Store (FAISS)**

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore

## **7. Similarity Search**

In [ ]:
results = vectorstore.similarity_search('What is AI?')

results

## **8. Create Retriever**

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

retriever.invoke('Explain machine learning')

## **9. Basic RAG Pipeline (Retriever + LLM)**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

qa_chain.invoke('Explain deep learning')

## **10. Custom Prompt for RAG**

In [ ]:
from langchain.prompts import PromptTemplate

template = '''
Use the following context to answer the question.

Context:
{context}

Question:
{question}
'''

prompt = PromptTemplate(
    template=template,
    input_variables=['context','question']
)


## **11. RAG using LCEL Pipeline**

In [ ]:
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {'context': retriever, 'question': lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke('What is machine learning?')

## **12. RAG with Sources**

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

result = qa_chain.invoke('Explain AI')

result['result']

## **13. Conversational RAG (Chat History)**

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever
)

conversation_chain.invoke({'question':'What is AI?', 'chat_history':[]})

## **14. Streaming RAG Responses**

In [ ]:
for chunk in rag_chain.stream('Explain neural networks'):
    print(chunk, end='', flush=True)